In [1]:
!pip install -q -U datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00


In [5]:
import os
from google.colab import files

os.makedirs("uzbek_ner_bio", exist_ok=True)

print("In the picker, select ALL THREE files together (Cmd-click or Cmd+A):")
up = files.upload()          # data-00000-of-00001.arrow, dataset_info.json, state.json

def canonical(name):
    if name.endswith(".arrow"):
        return "data-00000-of-00001.arrow"
    if "dataset_info" in name:
        return "dataset_info.json"
    if "state" in name:
        return "state.json"
    return None

for name, data in up.items():
    target = canonical(name)
    if target is None:
        print("skipped unexpected file:", name)
        continue
    with open(f"uzbek_ner_bio/{target}", "wb") as f:
        f.write(data)
    if os.path.exists(name):
        os.remove(name)      # clean the loose copy Colab also saved
    print(f"{name}  ->  uzbek_ner_bio/{target}  ({len(data):,} bytes)")

print()
print("folder contents:", sorted(os.listdir("uzbek_ner_bio")))

In the picker, select ALL THREE files together (Cmd-click or Cmd+A):


Saving data-00000-of-00001.arrow to data-00000-of-00001 (1).arrow
Saving dataset_info.json to dataset_info (1).json
Saving state.json to state (1).json
data-00000-of-00001 (1).arrow  ->  uzbek_ner_bio/data-00000-of-00001.arrow  (29,221,624 bytes)
dataset_info (1).json  ->  uzbek_ner_bio/dataset_info.json  (344 bytes)
state (1).json  ->  uzbek_ner_bio/state.json  (247 bytes)

folder contents: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']


In [6]:
from datasets import load_from_disk

bio = load_from_disk("uzbek_ner_bio")
print(bio)
print(bio[0]["tokens"][:8])
print(bio[0]["tags"][:8])

Dataset({
    features: ['tokens', 'tags'],
    num_rows: 19609
})
['Shvetsiya', 'hukumati', 'Stokholmdagi', 'asosiy', 'piyodalar', "ko'chasi", 'Drottninggatanda', 'odamlar']
['B-GPE', 'O', 'B-LOC', 'O', 'O', 'O', 'B-LOC', 'O']


In [7]:
CLASSES = ["PERSON", "ORG", "GPE", "LOC", "DATE"]
label_list = ["O"] + [f"{p}-{c}" for c in CLASSES for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(len(label_list), "labels:")
print(label_list)

11 labels:
['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']


In [8]:
from datasets import DatasetDict
tmp = bio.train_test_split(test_size=0.2, seed=42)
dev = tmp["test"].train_test_split(test_size=0.5, seed=42)
ds = DatasetDict(
train=tmp["train"],
validation=dev["train"],
test=dev["test"],
)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 15687
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
})


In [10]:
from collections import Counter
def b_share(split):
    c, tot = Counter(), 0
    for tags in split["tags"]:
        c.update(tags)
        tot += len(tags)
    return {cl: round(c["B-" + cl] / tot * 100, 2) for cl in CLASSES}
for name in ds:
    print(f"{name:11}", b_share(ds[name]))

train       {'PERSON': 1.73, 'ORG': 1.82, 'GPE': 2.93, 'LOC': 1.27, 'DATE': 0.65}
validation  {'PERSON': 1.78, 'ORG': 1.91, 'GPE': 2.89, 'LOC': 1.26, 'DATE': 0.61}
test        {'PERSON': 1.7, 'ORG': 1.73, 'GPE': 2.95, 'LOC': 1.23, 'DATE': 0.61}


In [11]:
from transformers import AutoTokenizer
MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)
print("fast tokenizer:", tok.is_fast)
ex = ds["train"][0]
enc = tok(ex["tokens"], is_split_into_words=True)
pieces = tok.convert_ids_to_tokens(enc["input_ids"])
word_ids = enc.word_ids()
for p, w in list(zip(pieces, word_ids))[:14]:
  src = ex["tokens"][w] if w is not None else "(special)"
  print(f"{p:20} word_id={str(w):5} from: {src}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

fast tokenizer: True
<s>                  word_id=None  from: (special)
▁Federal             word_id=0     from: Federal
▁qi                  word_id=1     from: qidiruv
dir                  word_id=1     from: qidiruv
uv                   word_id=1     from: qidiruv
▁by                  word_id=2     from: byurosi
uros                 word_id=2     from: byurosi
i                    word_id=2     from: byurosi
▁direktori           word_id=3     from: direktori
▁Jy                  word_id=4     from: Jyeyms
e                    word_id=4     from: Jyeyms
ym                   word_id=4     from: Jyeyms
s                    word_id=4     from: Jyeyms
▁Kom                 word_id=5     from: Komi


In [13]:
import numpy as np
def tok_len(batch):
    enc = tok(batch["tokens"], is_split_into_words=True)
    return {"n_tok": [len(x) for x in enc["input_ids"]]}
lens = np.array(ds["train"].map(tok_len, batched=True,
remove_columns=ds["train"].column_names)["n_tok"])
words = np.array([len(t) for t in ds["train"]["tokens"]])
print("tokens/row: min", lens.min(),
" p50/p90/p99/p99.5", np.percentile(lens, [50, 90, 99,
99.5]).astype(int),
" max", lens.max())
print("pieces per word (median):", round(np.percentile(lens, 50) /
np.percentile(words, 50), 2))

Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

tokens/row: min 28  p50/p90/p99/p99.5 [196 253 289 303]  max 361
pieces per word (median): 2.2


In [15]:
span_first_pos = [] # first-piece position of
# every B- span in train
for ex in ds["train"]:
    enc = tok(ex["tokens"], is_split_into_words=True)
    wid = enc.word_ids()
    first = {}
    for pos, w in enumerate(wid):
        if w is not None and w not in first:
            first[w] = pos
    for i, t in enumerate(ex["tags"]):
        if t.startswith("B-") and i in first:
            span_first_pos.append(first[i])

import numpy as np
sp = np.array(span_first_pos)
print("train spans measured:", len(sp))
for L in (128, 192, 256, 320):
    lost = int((sp >= L - 1).sum())
    print(f"max_length={L:3}: {lost:5} spans lost ({lost / len(sp) *
          100:.2f} %)")

train spans measured: 115106
max_length=128: 31528 spans lost (27.39 %)
max_length=192:  9003 spans lost (7.82 %)
max_length=256:   653 spans lost (0.57 %)
max_length=320:    23 spans lost (0.02 %)


In [17]:
MAXLEN = 256 # Define MAXLEN based on previous analysis (e.g., 256 or 320)
def tokenize_and_align(batch):
    enc = tok(batch["tokens"], is_split_into_words=True,
            truncation=True, max_length=MAXLEN)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        wid = enc.word_ids(batch_index=i)
        prev, row = None, []
        for w in wid:
            if w is None:
                row.append(-100) # specials
            elif w != prev:
                row.append(label2id[tags[w]]) # first piece of a word
            else:
                row.append(-100) # continuation pieces
            prev = w
        all_labels.append(row)
    enc["labels"] = all_labels
    return enc
tok_ds = ds.map(tokenize_and_align, batched=True,
remove_columns=["tokens", "tags"])
print(tok_ds)

Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15687
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1961
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1961
    })
})


In [19]:
ex = tok_ds["train"][0]
pieces = tok.convert_ids_to_tokens(ex["input_ids"])
for p, l in list(zip(pieces, ex["labels"]))[:16]:
    print(f"{p:20} {id2label[l] if l != -100 else '-100'}")

<s>                  -100
▁Federal             B-ORG
▁qi                  I-ORG
dir                  -100
uv                   -100
▁by                  I-ORG
uros                 -100
i                    -100
▁direktori           O
▁Jy                  B-PERSON
e                    -100
ym                   -100
s                    -100
▁Kom                 I-PERSON
i                    -100
▁San                 B-GPE


In [20]:
from transformers import DataCollatorForTokenClassification
collator = DataCollatorForTokenClassification(tok)
batch = collator([tok_ds["train"][i] for i in range(2)])
print({k: tuple(v.shape) for k, v in batch.items()})
print("tail of labels row 0:", batch["labels"][0][-6:].tolist())

{'input_ids': (2, 238), 'attention_mask': (2, 238), 'labels': (2, 238)}
tail of labels row 0: [0, -100, -100, 0, -100, -100]


In [21]:
import json, shutil
from google.colab import files
ds.save_to_disk("uzbek_ner_splits")
tok_ds.save_to_disk("uzbek_ner_tokenized")
json.dump(label_list, open("label_list.json", "w"), ensure_ascii=False)
shutil.make_archive("uzbek_ner_splits", "zip", "uzbek_ner_splits")
shutil.make_archive("uzbek_ner_tokenized", "zip", "uzbek_ner_tokenized")
files.download("uzbek_ner_splits.zip")
files.download("uzbek_ner_tokenized.zip")
files.download("label_list.json")

Saving the dataset (0/1 shards):   0%|          | 0/15687 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1961 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1961 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15687 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1961 [00:00<?, ? examples/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>